# Ancient Greek MMS voice smoke test
Runs Meta's dedicated `facebook/mms-tts-grc` checkpoint and produces a few WAV clips for pronunciation review.

The checkpoint is CC-BY-NC 4.0; use this notebook for the non-commercial prototype/evaluation stage.

In [ ]:
!pip -q install 'transformers>=4.45,<5' accelerate safetensors


In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, VitsModel
from IPython.display import Audio, display
from scipy.io.wavfile import write

MODEL = 'facebook/mms-tts-grc'
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = VitsModel.from_pretrained(MODEL).eval()
print('sample rate:', model.config.sampling_rate)


In [ ]:
def speak(text, filename='ancient-greek.wav', seed=7):
    torch.manual_seed(seed)
    inputs = tokenizer(text.lower(), return_tensors='pt')
    with torch.inference_mode():
        audio = model(**inputs).waveform.squeeze().cpu().numpy()
    audio = audio / max(1.0, float(np.max(np.abs(audio))))
    write(filename, model.config.sampling_rate, (audio * 32767).astype(np.int16))
    display(text)
    display(Audio(audio, rate=model.config.sampling_rate))
    return filename


In [ ]:
speak('ὁ Δικαιόπολις αὐτουργός ἐστιν.', 'dikaiopolis.wav')


## Classical-vs-Modern audit
Listen specifically for stop consonants, vowel distinctions, diphthongs, aspiration, rough breathing, and geminates.

In [ ]:
AUDIT = {
  'stops': 'βίος, γένος, δῶρον',
  'eta_vs_iota': 'ἡμέρα, ἱερός',
  'omega_vs_omicron': 'δῶρον, λόγος',
  'upsilon': 'Κῦρος, ὕδωρ',
  'ai': 'παῖς, αἰεί',
  'oi': 'οἶκος, οἶνος',
  'ei_ou': 'εἶμι, οὐρανός',
  'au_eu': 'αὐτός, εὖ',
  'aspirates': 'θεός, φίλος, χρόνος',
  'rough_breathing': 'ὁ, ἡ, ἥλιος',
  'gamma_nasal': 'ἄγγελος, ἀνάγκη',
  'geminates': 'θάλαττα, ἄλλος',
}
for i, (name, text) in enumerate(AUDIT.items(), 1):
    print('\n###', name)
    speak(text, f'{i:02d}-{name}.wav', seed=7+i)
